# Simulating Circuits Locally with the Neutral Atom Simulator

The QDK includes a local **neutral atom device simulator** that models the behavior of neutral-atom quantum hardware, including qubit loss noise. This makes it useful for testing circuits and understanding noisy results before submitting to real hardware on Azure Quantum.

This notebook shows how to:
- Run circuits against the local simulator using both **Qiskit** (`NeutralAtomBackend`) and **Cirq** (`NeutralAtomSampler`)
- Configure qubit loss noise via `NoiseConfig`
- Interpret results that include loss markers, using the **accepted** (clean) and **raw** (all shots) result fields

## Prerequisites

Install the `qdk` package with Qiskit and Cirq support:

In [ ]:
%pip install "qdk[qiskit,cirq]" matplotlib

After installing, restart the kernel if it was already running. Then verify imports:

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qdk.qiskit import NeutralAtomBackend

import cirq
from qdk.cirq import NeutralAtomSampler

from qdk.simulation import NoiseConfig

import matplotlib.pyplot as plt
from collections import Counter

print("Imports successful.")

## The circuit: 3-qubit GHZ state

We use a 3-qubit GHZ circuit throughout this notebook. It produces the maximally entangled state:

$$|\text{GHZ}\rangle = \frac{1}{\sqrt{2}}(|000\rangle + |111\rangle)$$

This is a good noise demonstration circuit because:
- Ideal results are simple: approximately 50% `000` and 50% `111`
- Any other outcome or missing qubit clearly indicates noise
- With 3 qubits, each shot has three independent chances to experience qubit loss

We define both a Qiskit and a Cirq version of the same circuit below.

In [ ]:
n_qubits = 3
SHOTS = 500
SEED = 42

---
## Part A: Qiskit

### Build and transpile the circuit

The `NeutralAtomBackend` natively supports the gate set `{Rz, SX, CZ}`. Circuits written with higher-level gates (like `H` and `CX`) must first be transpiled into this native set.

We transpile manually with `skip_transpilation=True` on the subsequent `run()` calls. This gives us full control over the decomposition and avoids the backend re-transpiling on each run.

In [ ]:
# Build the high-level GHZ circuit
ghz_qiskit = QuantumCircuit(n_qubits, n_qubits)
ghz_qiskit.h(0)
for i in range(n_qubits - 1):
    ghz_qiskit.cx(i, i + 1)
ghz_qiskit.measure(range(n_qubits), range(n_qubits))

print("High-level circuit:")
print(ghz_qiskit.draw())

# Transpile to the native gate set {Rz, SX, CZ}
backend = NeutralAtomBackend()
native_qiskit = transpile(ghz_qiskit, backend=backend)

print("\nNative gate circuit:")
print(native_qiskit.draw())

### Noiseless simulation

Run the native circuit with no noise configured. We expect to see only the two ideal GHZ outcomes: `000` and `111`.

In [ ]:
job_noiseless = backend.run(
    native_qiskit,
    shots=SHOTS,
    seed=SEED,
    skip_transpilation=True,
)
data_noiseless = job_noiseless.result().results[0].data

print("Noiseless counts:", dict(data_noiseless.counts))

fig, ax = plt.subplots(figsize=(6, 4))
plot_histogram(dict(data_noiseless.counts), ax=ax)
ax.set_title("Qiskit — Noiseless GHZ")
ax.yaxis.grid(True, linestyle="--", alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

### Configure qubit loss noise

`NoiseConfig` controls the per-gate noise parameters. Here we set an 8% qubit-loss probability on `Rz` gates.

Since the `H` gate decomposes to `Rz` gates in the native set, every qubit passes through at least one `Rz` and has a chance of being lost before measurement. When a qubit is lost, the simulator records `"-"` in its bitstring position rather than `"0"` or `"1"`.

In [ ]:
noise = NoiseConfig()
noise.rz.loss = 0.08  # 8% qubit-loss probability per Rz gate

### Noisy simulation and reading results

The result data (`job.result().results[0].data`) exposes two parallel sets of fields:

| Field | What it contains |
|---|---|
| **`counts`** | Bitstring → shot count, accepted shots only (no `"-"`) |
| **`probabilities`** | Bitstring → empirical probability, accepted shots only |
| **`memory`** | Per-shot bitstring list, accepted shots only |
| **`raw_counts`** | Bitstring → shot count, all shots (loss shots have `"-"`) |
| **`raw_probabilities`** | Bitstring → empirical probability, all shots |
| **`raw_memory`** | Per-shot bitstring list, all shots |

Use the **accepted** fields for downstream analysis. Use the **raw** fields to inspect or quantify loss.

In [ ]:
job_noisy = backend.run(
    native_qiskit,
    shots=SHOTS,
    noise=noise,
    seed=SEED,
    skip_transpilation=True,
)
data_noisy = job_noisy.result().results[0].data

accepted = dict(data_noisy.counts)
raw      = dict(data_noisy.raw_counts)

total_raw      = sum(raw.values())
total_accepted = sum(accepted.values())
total_lost     = total_raw - total_accepted

print(f"Total shots : {total_raw}")
print(f"Accepted    : {total_accepted}  ({100 * total_accepted / total_raw:.1f}%)")
print(f"Lost        : {total_lost}  ({100 * total_lost / total_raw:.1f}%)")
print()
print("Accepted counts:", accepted)
print("Raw counts     :", raw)

### Visualize: accepted vs raw

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_histogram(accepted, ax=axes[0])
plot_histogram(raw,      ax=axes[1])

for ax, title in zip(axes, [
    "Qiskit — Accepted shots (loss filtered out)",
    "Qiskit — Raw shots (loss bitstrings included)",
]):
    ax.set_title(title)
    ax.yaxis.grid(True, linestyle="--", alpha=0.7)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

---
## Part B: Cirq

### Build the circuit

The `NeutralAtomSampler` accepts standard Cirq circuits directly. It internally converts them to OpenQASM 3.0 before simulating, so no manual transpilation step is needed.

In [ ]:
qubits = cirq.LineQubit.range(n_qubits)

ghz_cirq = cirq.Circuit(
    cirq.H(qubits[0]),
    *[cirq.CNOT(qubits[i], qubits[i + 1]) for i in range(n_qubits - 1)],
    cirq.measure(*qubits, key="result"),
)

print(ghz_cirq)

### Noiseless simulation

Run with no noise and confirm only `000` and `111` appear.

In [ ]:
sampler = NeutralAtomSampler()
cirq_result_noiseless = sampler.run(ghz_cirq, repetitions=SHOTS)

# histogram() returns a Counter of integer bitmask → count; format as binary strings
cirq_noiseless_counts = {
    format(k, f"0{n_qubits}b"): v
    for k, v in cirq_result_noiseless.histogram(key="result").items()
}
print("Noiseless counts:", cirq_noiseless_counts)

def plot_bar(ax, counts, title):
    labels = sorted(counts.keys())
    values = [counts[k] for k in labels]
    bars = ax.bar(labels, values)
    ax.set_title(title)
    ax.set_xlabel("Outcome")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)
    ax.yaxis.grid(True, linestyle="--", alpha=0.7)
    ax.set_axisbelow(True)
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(), str(value),
            ha="center", va="bottom", fontsize=9,
        )

fig, ax = plt.subplots(figsize=(6, 4))
plot_bar(ax, cirq_noiseless_counts, "Cirq — Noiseless GHZ")
plt.tight_layout()
plt.show()

### Noisy simulation and reading results

Pass the same `NoiseConfig` to `NeutralAtomSampler`. The result object exposes:

| Field | What it contains |
|---|---|
| **`result.measurements[key]`** | NumPy int8 array of accepted shots only (no `"-"`), shape `(accepted_shots, n_qubits)` |
| **`result.raw_measurements()[key]`** | String array of all shots, `"-"` where a qubit was lost, same shape |
| **`result.raw_shots`** | The original shot objects as returned by the simulator |

Use `result.measurements` for analysis. Use `result.raw_measurements()` and `result.raw_shots` to inspect loss.

In [ ]:
noisy_sampler = NeutralAtomSampler(noise=noise, seed=SEED)
cirq_result_noisy = noisy_sampler.run(ghz_cirq, repetitions=SHOTS)

# Accepted: int8 array, shape = (accepted_shots, n_qubits)
accepted_arr = cirq_result_noisy.measurements["result"]

# Raw: string array, shape = (total_shots, n_qubits), "-" for lost qubits
raw_arr = cirq_result_noisy.raw_measurements()["result"]

cirq_total    = raw_arr.shape[0]
cirq_accepted = accepted_arr.shape[0]
cirq_lost     = cirq_total - cirq_accepted

print(f"Total shots : {cirq_total}")
print(f"Accepted    : {cirq_accepted}  ({100 * cirq_accepted / cirq_total:.1f}%)")
print(f"Lost        : {cirq_lost}  ({100 * cirq_lost / cirq_total:.1f}%)")

def arr_to_counts(arr):
    return Counter("".join(row) for row in arr)

cirq_accepted_counts = arr_to_counts(accepted_arr.astype(str))
cirq_raw_counts      = arr_to_counts(raw_arr)

print("\nAccepted counts:", dict(cirq_accepted_counts))
print("Raw counts     :", dict(cirq_raw_counts))

### Visualize: accepted vs raw

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_bar(axes[0], dict(cirq_accepted_counts), "Cirq — Accepted shots (loss filtered out)")
plot_bar(axes[1], dict(cirq_raw_counts),      "Cirq — Raw shots (loss bitstrings included)")
plt.tight_layout()
plt.show()

## Notes

- **`NoiseConfig` gate coverage**: Qubit-loss probability can be set independently per gate type (e.g. `noise.rz.loss`, `noise.sx.loss`, `noise.cz.loss`). You can mix different rates to model realistic hardware calibration profiles.
- **Why `skip_transpilation=True` (Qiskit only)**: Passing `skip_transpilation=True` tells the backend that the circuit is already in the native gate set. If you omit it, the backend will transpile automatically, but transpiling once and reusing saves time when running many shots or noise configurations.
- **Cirq result arrays**: Cirq's `measurements` and `raw_measurements()` return 2D NumPy arrays with one row per shot and one column per qubit. The `arr_to_counts` helper above joins each row into a single bitstring (e.g. `["1", "1", "1"]` → `"111"`) to make the distribution easy to inspect and visualize.
- **Next steps**: Once your circuit behaves as expected locally, submit it to real neutral-atom hardware on Azure Quantum using `cirq_submission_to_azure.ipynb` or `qiskit_submission_to_azure.ipynb`.
